# Day 16 Capstone — Predict Customer Churn
**End-to-end tabular machine learning project**

This notebook generates a reproducible telecom-style dataset, explores it, removes leakage, builds a preprocessing pipeline, compares four classifiers, tunes Gradient Boosting, evaluates the held-out test set, interprets drivers, and finishes with a business recommendation and reflection.

**Primary business priority:** protect recall for the churn class.  
**Model-selection metric:** ROC-AUC.  
**Data split:** 70% train, 15% validation, 15% test.

## Prerequisites — imports and reproducible synthetic data

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification

RNG = 42
rng = np.random.default_rng(RNG)
N = 4000

X_num, y = make_classification(
    n_samples=N,
    n_features=6,
    n_informative=5,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.76, 0.24],
    flip_y=0.13,
    class_sep=0.85,
    random_state=RNG
)

Xs = (X_num - X_num.mean(0)) / X_num.std(0)

df = pd.DataFrame()

# Numeric account features
df["tenure_months"] = np.clip(Xs[:, 4] * 12 + 34, 1, 72).round().astype(int)
df["monthly_charges"] = np.clip(Xs[:, 0] * 13 + 72, 18, 130).round(2)
df["support_tickets"] = np.clip(Xs[:, 2] * 1.7 + 3, 0, 25).round().astype(int)
df["avg_latency_ms"] = np.clip(Xs[:, 3] * 35 + 120, 20, 400).round().astype(int)
df["addons_count"] = np.clip(Xs[:, 1] * 1.3 + 4, 0, 8).round().astype(int)
df["days_since_login"] = np.clip(
    -Xs[:, 4] * 8 + 16 + rng.normal(0, 4, N), 0, 90
).round().astype(int)
df["total_charges"] = (
    df["monthly_charges"] * df["tenure_months"]
).round(2)

# Categorical plan features
def by_class(p_churn, p_stay, choices):
    out = np.empty(N, dtype=object)
    churn_mask = (y == 1)
    out[churn_mask] = rng.choice(choices, churn_mask.sum(), p=p_churn)
    out[~churn_mask] = rng.choice(choices, (~churn_mask).sum(), p=p_stay)
    return out

df["contract_type"] = by_class(
    [0.75, 0.18, 0.07], [0.34, 0.35, 0.31],
    ["Month-to-month", "One year", "Two year"]
)
df["payment_method"] = by_class(
    [0.55, 0.20, 0.15, 0.10], [0.25, 0.30, 0.25, 0.20],
    ["Electronic check", "Mailed check", "Bank transfer", "Credit card"]
)
df["internet_service"] = by_class(
    [0.65, 0.28, 0.07], [0.40, 0.35, 0.25],
    ["Fiber optic", "DSL", "None"]
)
df["has_dependents"] = by_class(
    [0.82, 0.18], [0.60, 0.40], ["No", "Yes"]
)

# Deliberate leakage feature: known only after churn has occurred
df["exit_survey_sent"] = np.where(
    y == 1,
    rng.binomial(1, 0.97, N),
    rng.binomial(1, 0.02, N)
)
df["churn"] = y

print("Dataset shape:", df.shape)
print("Churn rate:", round(df["churn"].mean(), 3))
display(df.head())

## Step 1 - Frame the problem and explore the data

**Problem statement:** Given a customer's account and plan features as of today, predict whether they will churn (`churn = 1`) so the retention team can target save-offers.

This is binary classification with an imbalanced target. Missing a customer who is about to leave can be more expensive than contacting a customer who would have stayed, so recall is an important business metric. ROC-AUC will be used for model selection.

In [ ]:
# Class balance
print("Class balance:")
print(df["churn"].value_counts(normalize=True).round(3), "\\n")

# Numeric feature means by churn class
num_cols = [
    "tenure_months", "monthly_charges", "support_tickets",
    "avg_latency_ms", "addons_count", "days_since_login",
    "total_charges"
]
print("Numeric feature means by churn:")
display(df.groupby("churn")[num_cols].mean().round(1).T)

# Churn rate by contract type
print("Churn rate by contract type:")
display(
    df.groupby("contract_type")["churn"]
      .mean().round(3).sort_values(ascending=False)
)

# Visualisations
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

for k, label in [(0, "Stay"), (1, "Churn")]:
    ax[0].hist(
        df.loc[df["churn"] == k, "tenure_months"],
        bins=25, alpha=0.6, label=label
    )

ax[0].set_title("Tenure by outcome")
ax[0].set_xlabel("Tenure (months)")
ax[0].set_ylabel("Number of customers")
ax[0].legend()

(df.groupby("contract_type")["churn"].mean()
   .sort_values().plot.barh(ax=ax[1]))
ax[1].set_title("Churn rate by contract")
ax[1].set_xlabel("Churn rate")

plt.tight_layout()
plt.show()

## Step 2 - Split train, validation and test; check leakage

In [ ]:
from sklearn.model_selection import train_test_split

# Identify numeric features suspiciously correlated with the target.
leak_scores = (
    df.corr(numeric_only=True)["churn"]
      .drop("churn").abs().sort_values(ascending=False)
)
print("Absolute correlation with churn:")
display(leak_scores.round(3))

suspects = leak_scores[leak_scores > 0.90].index.tolist()
print("Potential leakage columns:", suspects)

# Remove post-outcome leakage column(s).
df_clean = df.drop(columns=suspects)

X = df_clean.drop(columns=["churn"])
y = df_clean["churn"]

# 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RNG
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RNG
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

print(
    "Churn rate train/validation/test:",
    round(y_train.mean(), 3),
    round(y_val.mean(), 3),
    round(y_test.mean(), 3)
)

**Leakage reasoning:** `exit_survey_sent` is sent because a customer has already churned, so it would not be available at prediction time. `total_charges` stays because it is cumulative account information available today; a derived feature is not automatically leakage.

## Step 3 - Preprocessing with ColumnTransformer and Pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_features = X_train.select_dtypes(include="number").columns.tolist()
cat_features = X_train.select_dtypes(include="object").columns.tolist()

print("Numeric features:", num_features)
print("Categorical features:", cat_features)

pre = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ]
)

# Sanity check only: fit this diagnostic transformer on training data.
Z_train = pre.fit_transform(X_train)
print("Transformed training matrix shape:", Z_train.shape)
print("Feature count after one-hot encoding:", len(pre.get_feature_names_out()))

The preprocessing is included inside each model Pipeline below. Cross-validation therefore fits the scaler and encoder separately on each training fold, preventing validation-fold statistics from leaking into model fitting. The diagnostic `pre.fit_transform` above is only for checking the transformed feature count.

## Step 4 - Train and compare four model families

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

models = {
    "LogReg": LogisticRegression(
        max_iter=2000, class_weight="balanced"
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, random_state=RNG, class_weight="balanced"
    ),
    "GradBoost": GradientBoostingClassifier(random_state=RNG),
    "SVM_RBF": SVC(
        kernel="rbf", probability=True,
        class_weight="balanced", random_state=RNG
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)
results = {}

for name, clf in models.items():
    pipe = Pipeline([
        ("pre", pre),
        ("clf", clf)
    ])
    scores = cross_val_score(
        pipe, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1
    )
    results[name] = scores
    print(
        f"{name:13s} ROC-AUC = "
        f"{scores.mean():.3f} +/- {scores.std():.3f}"
    )

comparison = pd.DataFrame({
    name: {
        "mean_roc_auc": scores.mean(),
        "std_roc_auc": scores.std()
    }
    for name, scores in results.items()
}).T.sort_values("mean_roc_auc", ascending=False)

display(comparison)

We will carry Gradient Boosting into tuning and keep Logistic Regression as an interpretable baseline. If the scores are close, model simplicity, training cost, maintainability and explainability also matter.

## Step 5 - Tune Gradient Boosting with GridSearchCV

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score

gb_pipe = Pipeline([
    ("pre", pre),
    ("clf", GradientBoostingClassifier(random_state=RNG))
])

param_grid = {
    "clf__n_estimators": [150, 300],
    "clf__learning_rate": [0.05, 0.1],
    "clf__max_depth": [2, 3]
}

grid = GridSearchCV(
    estimator=gb_pipe,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    return_train_score=True
)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validation ROC-AUC:", round(grid.best_score_, 3))

best_model = grid.best_estimator_

val_proba = best_model.predict_proba(X_val)[:, 1]
val_auc = roc_auc_score(y_val, val_proba)
print("Validation ROC-AUC:", round(val_auc, 3))

The validation set is used for a confirmatory check and threshold selection. The test set remains untouched until the next step.

## Step 6 - Final test evaluation and threshold trade-off

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, roc_auc_score
)

# First and only evaluation of the held-out test set for final reporting.
proba = best_model.predict_proba(X_test)[:, 1]
pred_050 = (proba >= 0.50).astype(int)

print("TEST ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print()
print(classification_report(
    y_test, pred_050, target_names=["stay", "churn"], digits=3
))

cm = confusion_matrix(y_test, pred_050)
ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["stay", "churn"]
).plot(values_format="d")
plt.title("Confusion matrix at threshold 0.50")
plt.show()

# Threshold comparison on the held-out test set for descriptive reporting.
# For a production decision, choose the threshold using validation data,
# then evaluate that frozen threshold on test data.
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10} {'F1':>8}")
for threshold in [0.50, 0.40, 0.35, 0.30, 0.25]:
    pred = (proba >= threshold).astype(int)
    print(
        f"{threshold:10.2f} "
        f"{precision_score(y_test, pred):10.3f} "
        f"{recall_score(y_test, pred):10.3f} "
        f"{f1_score(y_test, pred):8.3f}"
    )

**Important methodology note:** The lab asks you to inspect the threshold trade-off. Strictly, the operating threshold should be selected using validation data, not by choosing the best-looking row on the test set. The test set should provide the final unbiased estimate after the threshold is frozen. The following cell demonstrates validation-based threshold selection and then reports its test performance.

In [ ]:
# Select a recall-oriented threshold on validation data.
threshold_candidates = [0.50, 0.40, 0.35, 0.30, 0.25]
threshold_rows = []

for threshold in threshold_candidates:
    val_pred = (val_proba >= threshold).astype(int)
    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_val, val_pred, zero_division=0),
        "recall": recall_score(y_val, val_pred, zero_division=0),
        "f1": f1_score(y_val, val_pred, zero_division=0)
    })

threshold_table = pd.DataFrame(threshold_rows)
display(threshold_table.round(3))

# Choose the threshold with the highest recall among candidates
# while keeping validation precision at or above 0.70.
eligible = threshold_table[threshold_table["precision"] >= 0.70]
if not eligible.empty:
    chosen_threshold = float(
        eligible.sort_values(["recall", "f1"], ascending=False)
                .iloc[0]["threshold"]
    )
else:
    chosen_threshold = 0.50

print("Chosen threshold from validation:", chosen_threshold)

# Final test evaluation at the frozen threshold
test_pred = (proba >= chosen_threshold).astype(int)
print("Test precision:", round(precision_score(y_test, test_pred), 3))
print("Test recall:", round(recall_score(y_test, test_pred), 3))
print("Test F1:", round(f1_score(y_test, test_pred), 3))
print("Test ROC-AUC:", round(roc_auc_score(y_test, proba), 3))

ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, test_pred),
    display_labels=["stay", "churn"]
).plot(values_format="d")
plt.title(f"Test confusion matrix at threshold {chosen_threshold:.2f}")
plt.show()

## Step 7 - Interpret the model

In [ ]:
from sklearn.inspection import permutation_importance

# Permutation importance on validation data
perm = permutation_importance(
    best_model, X_val, y_val,
    n_repeats=8,
    random_state=RNG,
    scoring="roc_auc"
)

imp = pd.Series(
    perm.importances_mean, index=X.columns
).sort_values(ascending=False)

print("Permutation importance (ROC-AUC decrease when shuffled):")
display(imp.round(4))

# Logistic Regression baseline for signed coefficients
logreg = Pipeline([
    ("pre", pre),
    ("clf", LogisticRegression(
        max_iter=2000, class_weight="balanced"
    ))
])
logreg.fit(X_train, y_train)

coef = pd.Series(
    logreg.named_steps["clf"].coef_[0],
    index=logreg.named_steps["pre"].get_feature_names_out()
)

print("Features with largest positive coefficients (push toward churn):")
display(coef.sort_values(ascending=False).head(6).round(3))

print("Features with largest negative coefficients (push away from churn):")
display(coef.sort_values(ascending=True).head(6).round(3))

imp.sort_values().plot.barh(figsize=(8, 5))
plt.title("Permutation importance")
plt.xlabel("Mean ROC-AUC decrease")
plt.tight_layout()
plt.show()

Permutation importance tells us which original input columns are useful to the tuned model. Logistic Regression coefficients show direction, but their magnitude is on the transformed feature scale and should not be interpreted as causal effects.

## Step 8 - Business recommendation and scorecard

In [ ]:
p_final = (proba >= chosen_threshold).astype(int)

summary = {
    "primary_metric": "Recall on churn; ROC-AUC for model selection",
    "chosen_model": "Tuned GradientBoostingClassifier",
    "test_roc_auc": round(roc_auc_score(y_test, proba), 3),
    "operating_threshold": chosen_threshold,
    "test_recall": round(recall_score(y_test, p_final), 3),
    "test_precision": round(precision_score(y_test, p_final), 3),
    "test_f1": round(f1_score(y_test, p_final), 3),
    "top_drivers": imp.head(3).index.tolist()
}

for key, value in summary.items():
    print(f"{key:22s}: {value}")